In [6]:
import mbuild as mb
from mbuild import Polymer
from gmso.core.forcefield import ForceField

gaff = ForceField("../files/gaff.xml")
oplsa = ForceField("oplsaa")

# Building polymers in mBuild
- Tagging bonding sites with SMILES strings:
    - This implementation is inspired by polymer syntax approaches such as BIGSmiles, CGSmiles and more.
    - It is not meant to be a full implementation of the above syntax approaches
    - You can put any tag value inside the `{}` brackets and carry it over to the polymer builder

In [6]:
monomer = mb.load("c1c{<}sc{>}c1", smiles=True)
chain = Polymer()
chain.add_monomer(monomer, head_tag=">", tail_tag="<", separation=0.145)
chain.build(n=4)
chain.visualize().show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [4]:
peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_chain = Polymer()
peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
peg_chain.build(n=34)
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Correct bonding graph, but bad configurations
- By using tagged SMILES strings, and the Polymer() class in mBuild, we can easily generate the correct polymer bonding topology
- However, we have no control over the chain configuraiton
- Try: Change `n` to a large number (40 - 50) in the thiophene example
- Challenges: Long straight chains create challenges in building polymer systems

In [7]:
from mbuild.path import Path, straight_line, lamellar, cyclic, hard_sphere_random_walk, lamellar, knot

In [8]:
line_path = straight_line(N=20, spacing=0.40)
line_path.visualize(radius=0.40)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
lamellar_2D = lamellar(spacing=0.45, layer_length=5, layer_separation=1.2, num_layers=5)
lamellar_2D.visualize(radius=0.40)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
lamellar_3D = lamellar(spacing=0.45, layer_length=5, layer_separation=1.2, num_layers=5, stack_separation=1.8, num_stacks=4)
lamellar_3D.visualize(radius=0.40)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [11]:
ring = cyclic(N=30, spacing=0.45)
ring.visualize(radius=0.40)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Coarse-grained chains
As we can see, the structures generated by all of these different "paths" are essentially coarse-grained polymers

Using mBuild polymer builder to back-map

In [11]:
ring = cyclic(N=30, spacing=0.30)

peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_chain = Polymer()
peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
peg_chain.build_from_path(path=ring, bond_head_tail=True, add_hydrogens=False)
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [12]:
lamellar_2D = lamellar(spacing=0.45, layer_length=5, layer_separation=1.2, num_layers=5)
monomer = mb.load("c1c{<}sc{>}c1", smiles=True)
chain = Polymer()
chain.add_monomer(monomer, head_tag=">", tail_tag="<", separation=0.145)
chain.build_from_path(lamellar_2D)
chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
lamellar_3D = lamellar(spacing=0.40, layer_length=5, layer_separation=1.0, num_layers=5, stack_separation=1.5, num_stacks=4)
monomer = mb.load("c1c{<}sc{>}c1", smiles=True)
chain = Polymer()
chain.add_monomer(monomer, head_tag=">", tail_tag="<", separation=0.145)
chain.build_from_path(lamellar_3D)
chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [17]:
random_walk_path = hard_sphere_random_walk(radius=0.30, bond_length=0.301, rw_angles=(1.5, 3.14), termination=25)
random_walk_path.visualize(radius=0.30)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [18]:
peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_chain = Polymer()
peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
peg_chain.build_from_path(path=random_walk_path)
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Backmapping and Energy Minimization

In [13]:
from mbuild.simulation import HoomdSimulation, hoomd_fire, hoomd_cap_displacement, hoomd_nvt, ForcesHandler

from gmso.core.forcefield import ForceField
from gmso.parameterization.parameterize import apply

gaff = ForceField("../files/gaff.xml")
oplsaa = ForceField("oplsaa")

In [10]:
peg_monomer = mb.load("C{<}CO{>}", smiles=True)
print(peg_monomer.volume()**(1/3))
spacing = peg_monomer.volume()**(1/3)

0.37833062717870447


In [14]:
random_walk_path = hard_sphere_random_walk(radius=0.30, bond_length=0.301, rw_angles=(1.5, 3.14), termination=25)

peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_chain = Polymer()
peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
peg_chain.build_from_path(path=random_walk_path)
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [17]:
hoomd_sim = HoomdSimulation(compound=peg_chain, forcefield=gaff, r_cut=1.2)
hoomd_cap_displacement(
    n_steps=1000,
    compound=peg_chain,
    sim=hoomd_sim,
    forces_handler=ForcesHandler(),
    max_displacement=1e-3,
    dt=1
)

hoomd_fire(compound=peg_chain, n_iterations=5, n_steps=1000, sim=hoomd_sim, forces_handler=ForcesHandler())
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [20]:
peg_chain.add(random_walk_path.to_compound())
peg_chain.visualize(bead_size=0.5)

2026-06-30 15:47:36,343 - mbuild.conversion - WARNING - No element attribute associated with '<_A pos=([ 0.1644 -0.0367  0.2152]), 1 bonds, id: 13414489632>'; and no matching elements found based upon the compound name. Setting atomic number to zero.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Exercise: 
- Build a PEG chain from a path (random walk, lamellar 2D or 3D, cyclic, knot)
- Design and run your own energy minimization protocol. Choose between the GAFF and OPLSAA forcefields

Tip: Use the `help` function on the imports below to see their parameters and doc strings.

e.g., `help(knot)`


In [21]:
from mbuild.path import (
    lamellar,
    cyclic,
    straight_line,
    knot,
    hard_sphere_random_walk,
    zigzag,
)

gaff = ForceField("../files/gaff.xml")
oplsa = ForceField("oplsaa")

In [22]:
# build instance of a path
# Create a PEG monomer, with tagged SMILES indicating the bonding atoms
# Create polymer chain instance
# Use the .add_monomer() method with tagged smiles string for PEG
# Build the chain from the path

# Create a Hoomd simulation object
# Pass the resulting chain into hoomd_cap_displacement, hoomd_fire (you can try hoomd_nvt as well)

In [23]:
help(knot)

Help on function knot in module mbuild.path.build:

knot(spacing, N, m, path=None, closed=True, bead_name='_A')
    Generate a knot path.

    Parameters
    ----------
    spacing : float (nm)
        The spacing between sites along the path.
    N : int
        The number of total sites in the path.
    m : int in [3, 4, 5]
        The number of crossings in the knot.
        3 gives the trefoil knot, 4 gives the figure 8 knot and 5 gives the cinquefoil knot.
    path : mbuild.path.Path
        The Path object to populate with coordinates.
        Defaults to creating a new, empty Path if no Path is given.
    closed : bool, default True
        If `True` the cyclic path is closed by bonding the first and last sites together
    bead_name : str or path.BeadNamer, optional, default '_A'
        Name(s) to assign to beads. A plain string assigns the same name to
        every bead. Pass a ``BeadNamer`` instance for heterogeneous sequences.
        See mbuild.path.namers.py

